# Step 7: Tuning the ANN
Grid search: 3 dropouts x 2 learning rates x 2 sizes = 12 runs with Adam, plus 1 run with SGD. Settings are chosen on a **validation** set; the test set is used only once at the end (development stage, random split).
*Same results as `src/07_tuning.py`.*

**In short** (what this notebook shows)

- **What:** ANN tuning: 13 settings of layer sizes, dropout and learning rate, chosen on a validation set, then tested.
- **Needs the NoXi data?** Yes. Random split (development step).
- **Result / use:** Best: 512-256 neurons, dropout 0.1, Adam 0.001 (test 0.865). These settings are used in the final ANN.

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page (MultiMediate): https://multimediate.perceptualui.org/datasets/Dataset_NoXi/ (if it is offline, use the links below)
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017: https://eprints.nottingham.ac.uk/45827 (DOI https://doi.org/10.1145/3136755.3136780)
- Extension NoXi+J (2024): https://arxiv.org/abs/2409.13726
- Data access: University of Augsburg, Human-Centered AI lab - noxi@hcai.eu
- Used here: **24 conversations** (English 026, 027, 028, 029, 030, 058, 062, 063; French 001, 007, 009, 011, 013, 014, 020, 021; German 068, 070, 072, 073, 074, 075, 076, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
keras.utils.set_random_seed(42)

In [4]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Load the data and split 60 / 20 / 20** (train / validation / test). The scaler learns only from the training part.

In [5]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")
cols = []
for c in df.columns:
    if c.startswith("voice_") or c.startswith("face_"):
        cols.append(c)
X = df[cols].fillna(0).to_numpy("float32")
y = df["engagement"].to_numpy("float32")

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)          # 20% test
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)  # 20% validation
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")
print("train", len(X_train), "| val", len(X_val), "| test", len(X_test))

train 29091 | val 9697 | test 9698


**Train and score one setting** on the validation set. The seed is reset so every setting starts the same.

In [6]:
def run(dropout, lr, h1, h2, optimizer="adam", epochs=30):
    keras.utils.set_random_seed(42)
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(h1, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(h2, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(1),
    ])
    if optimizer == "adam":
        opt = keras.optimizers.Adam(learning_rate=lr)
    else:
        opt = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    model.compile(optimizer=opt, loss="mse")
    model.fit(X_train, y_train, epochs=epochs, batch_size=128, verbose=0)
    val_ccc = ccc(y_val, model.predict(X_val, verbose=0).flatten())
    return val_ccc, model

**Grid search: 12 settings with Adam.**

In [7]:
rows = []
for dropout in [0.1, 0.3, 0.5]:
    for lr in [0.001, 0.0005]:
        for h1, h2 in [(256, 128), (512, 256)]:
            val_ccc, _ = run(dropout, lr, h1, h2)
            rows.append({"dropout": dropout, "lr": lr, "hidden": f"{h1}-{h2}", "h1": h1, "h2": h2,
                         "optimizer": "adam", "val_ccc": val_ccc})
            print(f"dropout {dropout}  lr {lr}  hidden {h1}-{h2}  adam   val CCC {val_ccc:.3f}")

dropout 0.1  lr 0.001  hidden 256-128  adam   val CCC 0.833


dropout 0.1  lr 0.001  hidden 512-256  adam   val CCC 0.836


dropout 0.1  lr 0.0005  hidden 256-128  adam   val CCC 0.832


dropout 0.1  lr 0.0005  hidden 512-256  adam   val CCC 0.832


dropout 0.3  lr 0.001  hidden 256-128  adam   val CCC 0.820


dropout 0.3  lr 0.001  hidden 512-256  adam   val CCC 0.825


... (2 outputs left out)

dropout 0.5  lr 0.001  hidden 256-128  adam   val CCC 0.795


dropout 0.5  lr 0.001  hidden 512-256  adam   val CCC 0.784


dropout 0.5  lr 0.0005  hidden 256-128  adam   val CCC 0.754


dropout 0.5  lr 0.0005  hidden 512-256  adam   val CCC 0.784


**13th run:** the best setting with SGD instead of Adam (SGD needs a 10x larger learning rate).

In [8]:
best = rows[0]
for r in rows:
    if r["val_ccc"] > best["val_ccc"]:
        best = r
val_sgd, _ = run(best["dropout"], best["lr"] * 10, best["h1"], best["h2"], optimizer="sgd")
rows.append({"dropout": best["dropout"], "lr": best["lr"] * 10, "hidden": best["hidden"], "h1": best["h1"],
             "h2": best["h2"], "optimizer": "sgd", "val_ccc": val_sgd})
print(f"dropout {best['dropout']}  lr {best['lr'] * 10}  hidden {best['hidden']}  sgd    val CCC {val_sgd:.3f}")

dropout 0.1  lr 0.01  hidden 512-256  sgd    val CCC 0.772


**All 13 settings, best first.**

In [9]:
results = pd.DataFrame(rows).sort_values("val_ccc", ascending=False).reset_index(drop=True)
print(results[["dropout", "lr", "hidden", "optimizer", "val_ccc"]].round(4).to_string(index=False))

 dropout     lr  hidden optimizer  val_ccc
     0.1 0.0010 512-256      adam   0.8357
     0.1 0.0010 256-128      adam   0.8331
     0.1 0.0005 512-256      adam   0.8319
     0.1 0.0005 256-128      adam   0.8319
     0.3 0.0010 512-256      adam   0.8246
     0.3 0.0010 256-128      adam   0.8198
     0.3 0.0005 512-256      adam   0.8126
     0.3 0.0005 256-128      adam   0.8091
     0.5 0.0010 256-128      adam   0.7950
     0.5 0.0010 512-256      adam   0.7841
     0.5 0.0005 512-256      adam   0.7839
     0.1 0.0100 512-256       sgd   0.7717
     0.5 0.0005 256-128      adam   0.7543


**Retrain the winner for 60 epochs and use the test set once.**

In [10]:
w = results.iloc[0]
val_ccc, model = run(float(w["dropout"]), float(w["lr"]), int(w["h1"]), int(w["h2"]), str(w["optimizer"]), epochs=60)
test_ccc = ccc(y_test, model.predict(X_test, verbose=0).flatten())
print("best setting:", w["hidden"], "| dropout", w["dropout"], "| lr", w["lr"], "|", w["optimizer"])
print(f"validation CCC {val_ccc:.3f} | TEST CCC {test_ccc:.3f}")

best setting: 512-256 | dropout 0.1 | lr 0.001 | adam
validation CCC 0.842 | TEST CCC 0.865


**Save the results** (`reports/tuning_results.csv`).

In [11]:
results[["dropout", "lr", "hidden", "optimizer", "val_ccc"]].to_csv("reports/tuning_results.csv", index=False)
print("saved reports/tuning_results.csv")

saved reports/tuning_results.csv
